# 🧬 Dark Manifold v4 — Direct Dynamics + Delta Loss

## Lessons Learned

| Version | Approach | Result |
|---------|----------|--------|
| v2 | MSE loss | Flat predictions (delta corr ~0) |
| v3 | Hamiltonian + delta loss | **Opposite** dynamics (delta corr -0.6) |
| **v4** | **Direct dynamics + delta loss** | Should work! |

## v4 Approach

- **No Hamiltonian** — too constrained, fights the signal
- **Direct dM/dt prediction** — network predicts rate of change
- **Delta loss** — forces learning actual dynamics
- **Stoichiometry constraint** — dM = S @ flux

In [ ]:
#@title 1️⃣ Upload Data
from google.colab import files
import pickle
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import re
from tqdm import tqdm

print("Upload syn3a_real_data.pkl:")
uploaded = files.upload()

with open('syn3a_real_data.pkl', 'rb') as f:
    raw_data = pickle.load(f)

print("Loaded!")

In [ ]:
#@title 2️⃣ Parse Real Data

dfs = raw_data['all_dataframes']
rxns_df = pd.DataFrame(dfs['reconstruction.xlsx:Reactions'])
mets_df = pd.DataFrame(dfs['reconstruction.xlsx:Metabolites'])

metabolites = mets_df['Metabolite ID'].tolist()
met_to_idx = {m: i for i, m in enumerate(metabolites)}

genes = set()
gene_to_rxns = {}
gpr_rules = rxns_df['GPR rule'].tolist()

for j, gpr in enumerate(gpr_rules):
    if pd.isna(gpr):
        continue
    gene_ids = re.findall(r'MMSYN1_\d+', str(gpr))
    for g in gene_ids:
        genes.add(g)
        if g not in gene_to_rxns:
            gene_to_rxns[g] = []
        gene_to_rxns[g].append(j)

genes = sorted(list(genes))
gene_to_idx = {g: i for i, g in enumerate(genes)}

n_genes = len(genes)
n_mets = len(metabolites)
n_rxns = len(rxns_df)

S = np.zeros((n_mets, n_rxns))
for j, row in rxns_df.iterrows():
    equation = row['Reaction equation']
    for i, met in enumerate(mets_df['Metabolite name'].tolist()):
        met_id = mets_df.iloc[i]['Metabolite ID']
        if met_id in met_to_idx and met in equation:
            idx = met_to_idx[met_id]
            left_part = equation.split('-->')[0] if '-->' in equation else equation.split('<==>')[0]
            S[idx, j] = -1 if met in left_part else +1

G = np.zeros((n_genes, n_rxns))
for g, rxn_indices in gene_to_rxns.items():
    g_idx = gene_to_idx[g]
    for r_idx in rxn_indices:
        G[g_idx, r_idx] = 1

print(f"Genes: {n_genes}, Metabolites: {n_mets}, Reactions: {n_rxns}")

In [ ]:
#@title 3️⃣ Delta Loss (same as v3)

class DeltaLoss(nn.Module):
    """Loss that forces learning DYNAMICS."""
    
    def __init__(self, delta_weight=5.0, direction_weight=2.0):
        super().__init__()
        self.delta_weight = delta_weight
        self.direction_weight = direction_weight
    
    def forward(self, pred_next, true_next, true_curr):
        # State loss
        state_loss = F.mse_loss(pred_next, true_next)
        
        # Delta loss
        pred_delta = pred_next - true_curr
        true_delta = true_next - true_curr
        delta_loss = F.mse_loss(pred_delta, true_delta)
        
        # Direction loss (soft sign)
        pred_sign = torch.tanh(pred_delta * 10)
        true_sign = torch.tanh(true_delta * 10)
        direction_loss = F.mse_loss(pred_sign, true_sign)
        
        total = state_loss + self.delta_weight * delta_loss + self.direction_weight * direction_loss
        
        return total, {
            'state': state_loss.item(),
            'delta': delta_loss.item(),
            'direction': direction_loss.item(),
        }

print("✅ DeltaLoss defined")

In [ ]:
#@title 4️⃣ Direct Dynamics Model (NO Hamiltonian)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

class DarkManifoldV4(nn.Module):
    """
    Direct dynamics model.
    
    Key insight: Predict dM/dt DIRECTLY, don't constrain with Hamiltonian.
    
    Architecture:
    1. Gene encoder → enzyme activities
    2. Metabolite encoder → current state embedding
    3. Flux predictor → reaction fluxes
    4. dM/dt = S @ flux (stoichiometry constraint)
    5. M_next = M + dt * dM/dt
    """
    
    def __init__(self, n_genes, n_metabolites, n_reactions, S, G, hidden_dim=256):
        super().__init__()
        
        self.n_genes = n_genes
        self.n_metabolites = n_metabolites
        self.n_reactions = n_reactions
        
        self.register_buffer('S', torch.tensor(S, dtype=torch.float32))
        self.register_buffer('G', torch.tensor(G, dtype=torch.float32))
        
        # Gene encoder → enzyme activities (Vmax)
        self.gene_encoder = nn.Sequential(
            nn.Linear(n_genes, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, n_reactions),
            nn.Softplus(),  # Vmax > 0
        )
        
        # Metabolite encoder
        self.met_encoder = nn.Sequential(
            nn.Linear(n_metabolites, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
        )
        
        # Flux predictor: combines enzyme activity + metabolite state
        self.flux_predictor = nn.Sequential(
            nn.Linear(hidden_dim + n_reactions, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, n_reactions),
        )
        
        # Learnable Km (Michaelis constant)
        self.log_Km = nn.Parameter(torch.zeros(n_reactions))
        
        # Gene regulation
        self.W_reg = nn.Parameter(torch.randn(n_genes, n_genes) * 0.01)
        
        # Learnable time scale
        self.log_tau = nn.Parameter(torch.zeros(1))
    
    @property
    def Km(self):
        return torch.exp(self.log_Km).clamp(0.01, 100.0)
    
    @property
    def tau(self):
        return torch.exp(self.log_tau).clamp(0.001, 10.0)
    
    def forward(self, gene_expr, met_conc, dt=0.01):
        """
        Predict next metabolite concentrations.
        
        Returns:
            next_metabolites: (B, n_mets)
            flux: (B, n_rxns)
            dM_dt: (B, n_mets) rate of change
        """
        # Gene regulation
        reg = torch.tanh(gene_expr @ self.W_reg.T) * 0.1
        regulated = gene_expr + reg
        
        # Gene → Vmax (enzyme activity)
        Vmax = self.gene_encoder(regulated)  # (B, n_rxns)
        
        # Encode metabolites
        met_hidden = self.met_encoder(met_conc)  # (B, hidden)
        
        # Predict flux direction/magnitude
        flux_input = torch.cat([met_hidden, Vmax], dim=-1)
        flux_raw = self.flux_predictor(flux_input)  # (B, n_rxns)
        
        # Apply MM kinetics: v = Vmax * S / (Km + S)
        substrate_proxy = met_conc.mean(dim=-1, keepdim=True) + 0.1
        mm_factor = substrate_proxy / (self.Km.mean() + substrate_proxy)
        
        flux = Vmax * flux_raw * mm_factor  # (B, n_rxns)
        
        # dM/dt = S @ flux (stoichiometry)
        dM_dt = flux @ self.S.T  # (B, n_mets)
        
        # Scale by learnable tau
        dM_dt = dM_dt * self.tau
        
        # Euler step
        next_met = met_conc + dt * dM_dt
        next_met = next_met.clamp(min=0)  # Concentrations ≥ 0
        
        return {
            'next_metabolites': next_met,
            'flux': flux,
            'dM_dt': dM_dt,
            'Vmax': Vmax,
        }


model = DarkManifoldV4(
    n_genes=n_genes,
    n_metabolites=n_mets,
    n_reactions=n_rxns,
    S=S,
    G=G,
    hidden_dim=256,
).to(device)

print(f"\nDarkManifoldV4 (Direct Dynamics):")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
#@title 5️⃣ Trajectory Generator

class TrajectoryGenerator:
    def __init__(self, S, G, n_genes, n_mets, n_rxns):
        self.S = torch.tensor(S, dtype=torch.float32)
        self.G = torch.tensor(G, dtype=torch.float32)
        self.n_genes = n_genes
        self.n_mets = n_mets
    
    def simulate(self, n_steps=50, dt=0.01, batch_size=1):
        gene_expr = torch.rand(batch_size, self.n_genes) * 2.0
        met_conc = torch.rand(batch_size, self.n_mets) * 2.0 + 0.1
        
        # Random trends
        trends = torch.randn(batch_size, self.n_mets) * 0.02
        
        enzyme = gene_expr @ self.G
        enzyme = F.softmax(enzyme, dim=-1)
        
        gene_traj = [gene_expr.clone()]
        met_traj = [met_conc.clone()]
        
        for _ in range(n_steps):
            substrate = met_conc.mean(dim=-1, keepdim=True)
            flux = enzyme * substrate / (0.5 + substrate)
            dM = flux @ self.S.T + trends + 0.001 * torch.randn_like(met_conc)
            
            met_conc = (met_conc + dt * dM).clamp(min=0.001)
            gene_expr = (gene_expr + 0.001 * torch.randn_like(gene_expr)).clamp(0.1, 3.0)
            
            gene_traj.append(gene_expr.clone())
            met_traj.append(met_conc.clone())
        
        return {
            'gene_trajectory': torch.stack(gene_traj, dim=1),
            'met_trajectory': torch.stack(met_traj, dim=1),
        }

generator = TrajectoryGenerator(S, G, n_genes, n_mets, n_rxns)
print("✅ Generator ready")

In [ ]:
#@title 6️⃣ Train!

n_epochs = 1000
batch_size = 64
n_steps = 50
lr = 1e-3

optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
loss_fn = DeltaLoss(delta_weight=5.0, direction_weight=2.0)  # Lower weights than v3

def train_step(batch):
    gene_traj = batch['gene_trajectory'].to(device)
    met_traj = batch['met_trajectory'].to(device)
    
    B, T, _ = gene_traj.shape
    total_loss = 0
    metrics = {'state': 0, 'delta': 0, 'direction': 0}
    
    for t in range(T - 1):
        out = model(gene_traj[:, t], met_traj[:, t])
        
        loss, m = loss_fn(
            out['next_metabolites'],
            met_traj[:, t + 1],
            met_traj[:, t],
        )
        total_loss += loss
        for k in metrics:
            metrics[k] += m[k]
    
    n = T - 1
    for k in metrics:
        metrics[k] /= n
    
    return total_loss / n, metrics

losses = []
print(f"Training: {n_epochs} epochs, batch={batch_size}")
print(f"Delta weight: {loss_fn.delta_weight}, Direction weight: {loss_fn.direction_weight}")

for epoch in tqdm(range(n_epochs)):
    batch = generator.simulate(n_steps=n_steps, batch_size=batch_size)
    
    optimizer.zero_grad()
    loss, metrics = train_step(batch)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step()
    
    losses.append(loss.item())
    
    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1}: Loss={loss.item():.4f} | "
              f"state={metrics['state']:.4f} delta={metrics['delta']:.4f} dir={metrics['direction']:.4f}")

print(f"\nFinal loss: {losses[-1]:.6f}")

In [ ]:
#@title 7️⃣ Evaluate
import matplotlib.pyplot as plt

model.eval()

test = generator.simulate(n_steps=50, batch_size=1)
gene_test = test['gene_trajectory'].to(device)
met_test = test['met_trajectory'].to(device)

# Predict
pred_traj = [met_test[:, 0]]
current = met_test[:, 0]

with torch.no_grad():
    for t in range(50):
        out = model(gene_test[:, t], current)
        current = out['next_metabolites']
        pred_traj.append(current)

pred_traj = torch.stack(pred_traj, dim=1)

true = met_test[0].cpu().numpy()
pred = pred_traj[0].cpu().numpy()

# Metrics
corr = np.corrcoef(true.flatten(), pred.flatten())[0, 1]

true_delta = true[1:] - true[:-1]
pred_delta = pred[1:] - pred[:-1]
delta_corr = np.corrcoef(true_delta.flatten(), pred_delta.flatten())[0, 1]

print(f"State Correlation: {corr:.4f}")
print(f"Delta Correlation: {delta_corr:.4f}  ← KEY METRIC")

# Plot
fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for i, ax in enumerate(axes.flatten()):
    ax.plot(true[:, i], 'b-', label='True', alpha=0.7)
    ax.plot(pred[:, i], 'r--', label='Predicted', alpha=0.7)
    ax.set_title(f'Metabolite {i}')
    ax.legend()

plt.suptitle(f'v4 Direct Dynamics | Delta Corr: {delta_corr:.3f}')
plt.tight_layout()
plt.savefig('trajectory_v4.png', dpi=150)
plt.show()

if delta_corr > 0.3:
    print("\n✅ Model learned dynamics!")
elif delta_corr > 0:
    print("\n⚠️ Some dynamics learned, needs more training")
else:
    print("\n❌ Dynamics not learned correctly")

In [ ]:
#@title 8️⃣ Save

save_dict = {
    'model_state_dict': model.state_dict(),
    'genes': genes,
    'metabolites': metabolites,
    'stoichiometry': S,
    'gene_reaction_map': G,
    'training_losses': losses,
    'version': 'v4_direct_dynamics',
}

torch.save(save_dict, 'dark_manifold_v4_direct.pt')

from google.colab import files
files.download('dark_manifold_v4_direct.pt')
files.download('trajectory_v4.png')